# Autoencoder Example 4 — Variational Autoencoder (VAE)

The basic autoencoder's latent space is a mess: decode a random code and you usually get
garbage, because nothing forced the codes to fill the space sensibly. A **Variational
Autoencoder** (Kingma & Welling, 2014) fixes this and turns the autoencoder into a proper
**generative model** you can sample from.

## Two changes from a plain autoencoder

1. **The encoder outputs a *distribution*, not a point.** For each image it predicts a
   mean $\mu$ and (log-)variance $\log\sigma^2$ of a Gaussian in latent space. We then
   **sample** a code $z = \mu + \sigma \odot \epsilon$, with $\epsilon \sim \mathcal{N}(0,I)$.
   (This 'reparameterization trick' keeps the sampling differentiable.)
2. **A KL term in the loss** pulls every encoded distribution toward the standard normal
   $\mathcal{N}(0, I)$. So the loss has two parts:

$$\mathcal{L} = \underbrace{\text{reconstruction error}}_{\text{decode well}} + \underbrace{\text{KL}\big(q(z|x)\,\|\,\mathcal{N}(0,I)\big)}_{\text{keep latent space tidy}}.$$

Because the latent space is now packed against $\mathcal{N}(0,I)$, we can **sample
$z \sim \mathcal{N}(0,I)$ and decode it to get a brand-new digit** — something a plain
autoencoder can't do reliably.

We use a **2D latent space** so we can actually *draw* it.

> Tested with `tensorflow==2.1`. Trains in ~1 minute on CPU.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

tf.random.set_seed(0); np.random.seed(0)
print('TensorFlow', tf.__version__)

In [ ]:
(x_train, _), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype('float32').reshape(-1, 784) / 255.0
x_test = x_test.astype('float32').reshape(-1, 784) / 255.0
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(60000).batch(256)
LATENT_DIM = 2
print('Train:', x_train.shape)

## 1. Encoder and decoder

The encoder has **two** output heads — `z_mean` and `z_log_var`. The decoder is an
ordinary network mapping a latent vector back to 784 pixels.

In [ ]:
enc_in = tf.keras.Input((784,))
h = layers.Dense(256, activation='relu')(enc_in)
h = layers.Dense(128, activation='relu')(h)
z_mean = layers.Dense(LATENT_DIM, name='z_mean')(h)
z_log_var = layers.Dense(LATENT_DIM, name='z_log_var')(h)
encoder = tf.keras.Model(enc_in, [z_mean, z_log_var], name='encoder')

dec_in = tf.keras.Input((LATENT_DIM,))
h = layers.Dense(128, activation='relu')(dec_in)
h = layers.Dense(256, activation='relu')(h)
dec_out = layers.Dense(784, activation='sigmoid')(h)
decoder = tf.keras.Model(dec_in, dec_out, name='decoder')

optimizer = tf.keras.optimizers.Adam(1e-3)

## 2. Training step (reparameterization + the two-part loss)

We sample `z = z_mean + exp(0.5*z_log_var) * eps`, decode it, then add the reconstruction
loss (per-pixel binary cross-entropy, summed over the image) and the closed-form KL
divergence between the encoder's Gaussian and the standard normal.

In [ ]:
@tf.function
def train_step(x):
    with tf.GradientTape() as tape:
        z_mean, z_log_var = encoder(x)
        eps = tf.random.normal(tf.shape(z_mean))
        z = z_mean + tf.exp(0.5 * z_log_var) * eps           # reparameterization trick
        x_hat = decoder(z)

        recon = 784.0 * tf.keras.losses.binary_crossentropy(x, x_hat)   # sum over pixels
        kl = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=1)
        loss = tf.reduce_mean(recon + kl)

    variables = encoder.trainable_variables + decoder.trainable_variables
    optimizer.apply_gradients(zip(tape.gradient(loss, variables), variables))
    return loss, tf.reduce_mean(recon), tf.reduce_mean(kl)

In [ ]:
EPOCHS = 25
for epoch in range(1, EPOCHS + 1):
    for batch in dataset:
        loss, recon, kl = train_step(batch)
    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:2d}/{EPOCHS}  loss={float(loss):.2f}  recon={float(recon):.2f}  kl={float(kl):.2f}')

## 3. The latent space, coloured by digit

Encode the test set and scatter each image's `z_mean`, coloured by its true label. Because
of the KL term the points form one tidy blob centred at the origin, with digits occupying
different regions.

In [ ]:
z_mean_test, _ = encoder.predict(x_test)
plt.figure(figsize=(6, 5))
sc = plt.scatter(z_mean_test[:, 0], z_mean_test[:, 1], c=y_test, cmap='tab10', s=4, alpha=0.6)
plt.colorbar(sc, label='digit'); plt.xlabel('z[0]'); plt.ylabel('z[1]')
plt.title('VAE latent space (test set)'); plt.show()

## 4. Generate new digits by walking the latent grid

Sweep a grid of `z` values (through the high-probability region of $\mathcal{N}(0,I)$) and
decode each one. The result is a smooth **manifold** of digits morphing into one another —
proof the latent space is continuous and meaningful.

In [ ]:
g = 15
try:
    import scipy.stats as st
    grid = st.norm.ppf(np.linspace(0.05, 0.95, g))   # spacing matched to N(0,1)
except Exception:
    grid = np.linspace(-2.5, 2.5, g)                 # fallback if scipy is absent

canvas = np.zeros((28 * g, 28 * g))
for i, zi in enumerate(grid[::-1]):
    for j, zj in enumerate(grid):
        img = decoder.predict(np.array([[zj, zi]], dtype='float32'))[0].reshape(28, 28)
        canvas[i*28:(i+1)*28, j*28:(j+1)*28] = img
plt.figure(figsize=(8, 8))
plt.imshow(canvas, cmap='gray'); plt.axis('off')
plt.title('Digits decoded across the 2D latent grid'); plt.show()

### Things to try

- Increase `LATENT_DIM` (e.g. 16) for sharper digits — you lose the easy 2D plot but gain
  quality.
- Weight the KL term (`beta * kl`) to build a **β-VAE** and study disentanglement.
- Make the encoder/decoder convolutional.
- Compare random samples here vs. the plain autoencoder's — the VAE's are far cleaner.